In [10]:
import pandas as pd

def cky_parse(tokens, grammar):
    n = len(tokens)
    # table[i][j] stores set of non-terminals covering tokens[i:j]
    table = [[set() for _ in range(n + 1)] for _ in range(n + 1)]

    # 1. Terminal rules (span = 1)
    for i, w in enumerate(tokens):
        table[i][i + 1] = set(grammar['lexical'].get(w, []))

    # 2. Binary rules (span lengths 2..n)
    for l in range(2, n + 1):
        for i in range(n - l + 1):
            j = i + l
            for k in range(i + 1, j):
                for B in table[i][k]:
                    for C in table[k][j]:
                        table[i][j].update(grammar['binary'].get((B, C), []))

    # Display table
    df = pd.DataFrame([[", ".join(table[i][j]) or "-" if j > i else ""
                        for j in range(1, n + 1)] for i in range(n)],
                      index=tokens, columns=[f"Span 1..{k}" for k in range(1, n + 1)])
    print("--- CKY Table ---\n", df)
    return 'S' in table[0][n]

# Test
grammar_cnf = {
    'lexical': {'the': ['Det'], 'cat': ['N'], 'dog': ['N'], 'chased': ['V']},
    'binary': {('Det', 'N'): ['NP'], ('V', 'NP'): ['VP'], ('NP', 'VP'): ['S']}
}
words = "the cat chased the dog".split()
print("CKY Valid:", cky_parse(words, grammar_cnf))

--- CKY Table ---
        Span 1..1 Span 1..2 Span 1..3 Span 1..4 Span 1..5
the          Det        NP         -         -         S
cat                      N         -         -         -
chased                             V         -        VP
the                                        Det        NP
dog                                                    N
CKY Valid: True


In [14]:
class State:
    def __init__(self, lhs, rhs, dot=0, origin=0):
        self.lhs, self.rhs, self.dot, self.origin = lhs, tuple(rhs), dot, origin

    def next_sym(self):
        return self.rhs[self.dot] if self.dot < len(self.rhs) else None

    def __eq__(self, o): return (self.lhs, self.rhs, self.dot, self.origin) == (o.lhs, o.rhs, o.dot, o.origin)
    def __hash__(self): return hash((self.lhs, self.rhs, self.dot, self.origin))

def earley_parse(tokens, grammar, start='S'):
    n = len(tokens)
    chart = [[] for _ in range(n + 1)]
    seen = [set() for _ in range(n + 1)]

    def add(state, k):
        if state not in seen[k]:
            seen[k].add(state)
            chart[k].append(state)

    add(State('GAMMA', [start]), 0)

    for k in range(n + 1):
        i = 0
        while i < len(chart[k]):
            st = chart[k][i]
            sym = st.next_sym()
            if sym is not None:
                if sym in grammar:  # Predictor
                    for rhs in grammar[sym]: add(State(sym, rhs, 0, k), k)
                elif k < n and sym == tokens[k]:  # Scanner
                    add(State(st.lhs, st.rhs, st.dot + 1, st.origin), k + 1)
            else:  # Completer
                for p in chart[st.origin]:
                    if p.next_sym() == st.lhs: add(State(p.lhs, p.rhs, p.dot + 1, p.origin), k)
            i += 1

    return any(st.lhs == 'GAMMA' and st.origin == 0 and st.next_sym() is None for st in chart[n])

# Test
cfg_grammar = {
    'S': [['NP', 'VP']], 'NP': [['Det', 'N']], 'VP': [['V', 'NP']],
    'Det': [['the']], 'N': [['cat'], ['dog']], 'V': [['chased']]
}
words = "the cat chased the dog".split()
print("Earley Valid:", earley_parse(words, cfg_grammar))

Earley Valid: True


In [15]:
def dep_parse(tokens, oracle_arcs):
    words = ["ROOT"] + tokens
    stack, buffer, arcs = [0], list(range(1, len(words))), []

    print(f"{'Stack':<25} | {'Buffer':<25} | {'Action'}")
    print("-" * 65)

    while buffer:
        st_top, buf_top = stack[-1] if stack else None, buffer[0]
        s_word, b_word = words[st_top] if st_top is not None else "", words[buf_top]

        # Arc matches based on oracle: (head, child, relation)
        left = any(h == b_word and c == s_word for h, c, _ in oracle_arcs)
        right = any(h == s_word and c == b_word for h, c, _ in oracle_arcs)

        if left and st_top != 0:
            rel = next(r for h, c, r in oracle_arcs if h == b_word and c == s_word)
            arcs.append((b_word, rel, s_word))
            action = f"LEFT-ARC ({rel})"
            stack.pop()
        elif right:
            rel = next(r for h, c, r in oracle_arcs if h == s_word and c == b_word)
            arcs.append((s_word, rel, b_word))
            action = f"RIGHT-ARC ({rel})"
            stack.append(buffer.pop(0))
        else:
            action = "SHIFT"
            stack.append(buffer.pop(0))

        print(f"{str([words[i] for i in stack]):<25} | {str([words[i] for i in buffer]):<25} | {action}")

    print("\nParsed Arcs:", arcs)

# Test
oracle = [("chased", "cat", "nsubj"), ("cat", "the", "det"), ("ROOT", "chased", "root"), ("chased", "dog", "dobj"), ("dog", "the", "det")]
dep_parse("the cat chased the dog".split(), oracle)

Stack                     | Buffer                    | Action
-----------------------------------------------------------------
['ROOT', 'the']           | ['cat', 'chased', 'the', 'dog'] | SHIFT
['ROOT']                  | ['cat', 'chased', 'the', 'dog'] | LEFT-ARC (det)
['ROOT', 'cat']           | ['chased', 'the', 'dog']  | SHIFT
['ROOT']                  | ['chased', 'the', 'dog']  | LEFT-ARC (nsubj)
['ROOT', 'chased']        | ['the', 'dog']            | RIGHT-ARC (root)
['ROOT', 'chased', 'the'] | ['dog']                   | SHIFT
['ROOT', 'chased']        | ['dog']                   | LEFT-ARC (det)
['ROOT', 'chased', 'dog'] | []                        | RIGHT-ARC (dobj)

Parsed Arcs: [('cat', 'det', 'the'), ('chased', 'nsubj', 'cat'), ('ROOT', 'root', 'chased'), ('dog', 'det', 'the'), ('chased', 'dobj', 'dog')]
